# Step 1: Setup prerequisites

### Set the LLM provider provided by your workshop instructor

In [ ]:
# LLM_PROVIDER can be set to one of "aws"/ "microsoft" / "google"
LLM_PROVIDER = "aws"

In [ ]:
import os
import sys
from pymongo import MongoClient

# Add parent directory to path to import from utils
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))
from utils import set_env

# ----- MONGODB SETUP -----
# If you are using your own MongoDB Atlas cluster, use the connection string for your cluster here
MONGODB_URI = os.environ.get("MONGODB_URI")
# Initialize a MongoDB Python client
mongodb_client = MongoClient(MONGODB_URI, appname="devrel-workshop-ai-agents")
# Check the connection to the server
mongodb_client.admin.command("ping")

# ----- [BACKUP] API KEY SETUP -----
# set_env("replace-with-passkey", [LLM_PROVIDER, "voyageai"])

# Step 2: Import data into MongoDB

In [ ]:
import json

### **Do not change the values assigned to the variables below**

In [ ]:
#  Database name
DB_NAME = "mongodb_genai_devday_agents"
# Name of the collection to store flights data
FLIGHTS_COLLECTION_NAME = "flights"
# Name of the collection to store AirBnB listings data
LISTINGS_COLLECTION_NAME = "listings"
# Name of the collection to store bookings data
BOOKINGS_COLLECTION_NAME = "bookings"
# Name of the collection to store user preferences
MEMORY_COLLECTION_NAME = "memories"

In [ ]:
# Connect to the `LISTINGS_COLLECTION_NAME` collection
listings_collection = mongodb_client[DB_NAME][LISTINGS_COLLECTION_NAME]
# Connect to the `FLIGHTS_COLLECTION_NAME` collection
flights_collection = mongodb_client[DB_NAME][FLIGHTS_COLLECTION_NAME]
# Connect to the `BOOKINGS_COLLECTION_NAME` collection
bookings_collection = mongodb_client[DB_NAME][BOOKINGS_COLLECTION_NAME]
# Connect to the `MEMORY_COLLECTION_NAME` collection
memory_collection = mongodb_client[DB_NAME][MEMORY_COLLECTION_NAME]

In [ ]:
# Clear out data written by previous runs of this notebook
# Documents left behind by an earlier run can conflict with the code below, for example
# memories that were saved with a different schema, or chat history for a thread ID reused below
print(f"Deleting existing documents from the {BOOKINGS_COLLECTION_NAME} and {MEMORY_COLLECTION_NAME} collections.")
bookings_collection.delete_many({})
memory_collection.delete_many({})
# Graph state is checkpointed to the `checkpointing_db` database by default (See Step 11)
checkpointing_db = mongodb_client["checkpointing_db"]
checkpointing_db["checkpoints"].delete_many({})
checkpointing_db["checkpoint_writes"].delete_many({})

In [ ]:
# Insert a dataset of AirBnB listings into the `LISTINGS_COLLECTION_NAME` collection
with open(f"../data/{LISTINGS_COLLECTION_NAME}.json", "r") as data_file:
    json_data = data_file.read()

data = json.loads(json_data)

print(f"Deleting existing documents from the {LISTINGS_COLLECTION_NAME} collection.")
listings_collection.delete_many({})
listings_collection.insert_many(data)
print(
    f"{listings_collection.count_documents({})} documents ingested into the {LISTINGS_COLLECTION_NAME} collection."
)

In [ ]:
# Insert a dataset of flight routes into the `FLIGHTS_COLLECTION_NAME` collection
with open(f"../data/{FLIGHTS_COLLECTION_NAME}.json", "r") as data_file:
    json_data = data_file.read()

data = json.loads(json_data)

print(f"Deleting existing documents from the {FLIGHTS_COLLECTION_NAME} collection.")
flights_collection.delete_many({})
flights_collection.insert_many(data)
print(
    f"{flights_collection.count_documents({})} documents ingested into the {FLIGHTS_COLLECTION_NAME} collection."
)

# Step 3: Instantiate the LLM

In [ ]:
from utils import get_llm

In [ ]:
# Obtain the Langchain LLM object using the `get_llm` function from the `utils` module
llm = get_llm(LLM_PROVIDER)

# Step 4: Create agent tools


In [ ]:
from langchain_mongodb.agent_toolkit.database import MongoDBDatabase
from langchain_mongodb.agent_toolkit.toolkit import MongoDBDatabaseToolkit

### Create a custom tool

In [ ]:
from datetime import datetime, timezone
from typing import Literal
from langchain.tools import tool

📚 https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html#pymongo.collection.Collection.insert_one

In [ ]:
# Define a custom tool to save booking details to MongoDB
@tool
def create_booking(
    booking_type: Literal["flight", "accommodation"],
    traveler_name: str,
    destination: str,
    item_name: str,
    price_usd: float,
    travelers: int = 1,
) -> str:
    """Save booking details to the bookings collection.

    Args:
        booking_type: Either "flight" or "accommodation".
        traveler_name: Full name of the traveler making the booking.
        destination: Destination city, e.g. "Barcelona".
        item_name: Flight number for flights, or listing name for accommodation.
        price_usd: Total price of this booking in USD.
        travelers: Number of travelers. Defaults to 1.
    """
    # Build the booking document to insert into MongoDB
    booking = {
        "booking_type": booking_type,
        "traveler_name": traveler_name,
        "destination": destination,
        "item_name": item_name,
        "price_usd": price_usd,
        "travelers": travelers,
        "created_at": datetime.now(timezone.utc),
    }
    # Insert the `booking` document into the `bookings` collection
    <CODE_BLOCK_1>
    return f"Booked {booking_type} to {destination}: {item_name}. Total price ${price_usd:.2f} for {travelers} travelers."

### Import tools from the MongoDB Database Toolkit

📚 https://langchain-mongodb.readthedocs.io/en/latest/langchain_mongodb/agent_toolkit/langchain_mongodb.agent_toolkit.toolkit.MongoDBDatabaseToolkit.html (See Instantiate)

In [ ]:
# Use the `from_connection_string` method of the `MongoDBDatabase` class and the `MONGODB_URI` defined in Step 1 to access the `DB_NAME` database
db = <CODE_BLOCK_2>

In [ ]:
# Initialize the MongoDB database toolkit with the `db` and `llm` defined previously
toolkit = <CODE_BLOCK_3>

### Investigate the tools

📚 https://langchain-mongodb.readthedocs.io/en/latest/langchain_mongodb/agent_toolkit/langchain_mongodb.agent_toolkit.toolkit.MongoDBDatabaseToolkit.html (See Tools)

In [ ]:
# Get the list of tools from the MongoDB database toolkit
tools = <CODE_BLOCK_4>
# Append the `create_booking` tool to the list of tools obtained from the MongoDB database toolkit
tools.append(create_booking)

In [ ]:
# Investigate the tool names, descriptions and arguments for each tool
tools_map = {t.name: t for t in tools}

for name, t in tools_map.items():
    print(f"{name}\n  description: {t.description}\n  args: {t.args}\n")

📚 https://docs.langchain.com/oss/python/langchain/models#invoke

In [ ]:
# Test out the `mongodb_list_collections` tool
# Access the `mongodb_list_collections` tool from the `tools_map` dictionary
# Use the `invoke` method to call the tool
# Refer to the output of the cell above to determine the arguments for the tool call
<CODE_BLOCK_5>

In [ ]:
# Test out the `mongodb_schema` tool on the `listings` collection
# Access the `mongodb_schema` tool from the `tools_map` dictionary
# Use the `invoke` method to call the tool
# Refer to the output of the cell above to determine the arguments for the tool call
<CODE_BLOCK_6>

In [ ]:
# Test the `mongodb_query_checker` tool
# The below tool call takes the MongoDB `query` as input, corrects it if necessary and returns the corrected query
query = 'db.flights.aggregate([{"$match": {"to_city": "Barcelona",}}, {"$limit": 3}])'
tools_map["mongodb_query_checker"].invoke(query)

In [ ]:
# Test the `mongodb_query` tool
# The below tool call executes the MongoDB `query` against the `flights` collection and returns the results
query = [
    {"$match": {"to_city": "Barcelona"}},
    {"$limit": 3}
]
tools_map["mongodb_query"].invoke(f"db.flights.aggregate({json.dumps(query)})")

In [ ]:
# Test the `create_booking` tool
# The below tool call writes a booking document to the `bookings` collection
create_booking.invoke({
    "booking_type": "flight",
    "traveler_name": "Sam Rivera",
    "destination": "Barcelona",
    "item_name": "TA303",
    "price_usd": 230.96,
    "travelers": 2
})

# Step 5: Create the LLM prompt

In addition to the tools, the MongoDB database toolkit also provides an LLM system prompt containing guidance on how to use the tools available in the toolkit. However, we will create our own system prompt since we want to provide additional tools and instructions to the agent.

You can access the system prompt available in the toolkit as follows:
```
from langchain_mongodb.agent_toolkit import MONGODB_AGENT_SYSTEM_PROMPT
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [ ]:
# Create a system prompt for the agent providing instructions on how to use the database tools
SYSTEM_PROMPT = """
You are an agent designed to interact with a MongoDB database.
You have access to the following tools: {tool_names}.
Only use the information returned by the tools to construct your final answer.

Most tools are read-only, but you can perform write actions to create new bookings and store them to MongoDB using the `create_booking` tool.

ALWAYS start by looking at the collections in the database to see what you can query, unless you have this information in memory.
Then you should query the schema of the most relevant collections.
To get data from MongoDB, create a syntactically correct MongoDB query to run, analyze the results of the query and decide what to do next.
You MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.
Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results.
You can order the results by a relevant field to return the most interesting examples in the database.
Never query for all the fields from a specific collection, only query for the relevant fields given the question.
Read queries MUST include the collection name and the contents of the aggregation pipeline. An example query looks like:

```python
db.flights.aggregate([{{"$match": {{"to_city": "Barcelona"}}}}, {{"$limit": 3}}])
```

Do not re-run tools unless absolutely necessary. If you are not able to get enough information using the tools, reply with I DON'T KNOW.
"""

In [ ]:
# Create a prompt template which includes the system prompt and a placeholder for the `messages` i.e. user queries and agent responses
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        MessagesPlaceholder(variable_name="messages")
    ]
)

In [ ]:
# Pre-fill `top_k` and `tool_names` in the prompt template
prompt = prompt.partial(top_k=5, tool_names=", ".join([tool.name for tool in tools]))

# Step 6: Give the LLM access to tools

📚 https://docs.langchain.com/oss/python/langgraph/quickstart#1-define-tools-and-model

In [ ]:
# Bind the `tools` defined in Step 4 to the `llm` instantiated in Step 3
bind_tools = <CODE_BLOCK_7>

📚 https://reference.langchain.com/python/langchain-core/runnables/base/Runnable/pipe (See Example)

In [ ]:
# Chain the `prompt` with the tool-augmented_llm using the `|` operator
llm_with_tools = <CODE_BLOCK_8>

In [ ]:
# Test that the LLM is making the right tool calls
llm_with_tools.invoke(
    ["I want to go to Barcelona. What's the cheapest nonstop flight there?"]
).tool_calls

The above test shows that the LLM will first call the `mongodb_list_collections` tool, given any query.

This is exactly what we want the agent to do as the first step so it can better understand our data.

# Step 7: Define graph state

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

In [ ]:
# Define the graph state
# We will track chat messages and the user's approval for write actions
class GraphState(TypedDict):
    messages: Annotated[list, add_messages]
    approved: bool

# Step 8: Define graph nodes

In [ ]:
from langchain_core.messages import ToolMessage
from langgraph.types import interrupt, Command
from typing import Dict, List

In [ ]:
# Define the agent node
def agent_node(state: GraphState) -> Dict[str, List]:
    """
    Agent node

    Args:
        state (GraphState): Graph state

    Returns:
        Dict[str, List]: Updates to the `messages` attribute of graph state
    """
    # Get the messages from the graph `state`
    messages = <CODE_BLOCK_9>
    # Invoke `llm_with_tools` with `messages` using the `invoke` method
    # HINT: See Step 6 for how to invoke `llm_with_tools`
    result = <CODE_BLOCK_10>
    # Write `result` to the `messages` attribute of the graph state
    return {"messages": [result]}

In [ ]:
# Define tool node
def tool_node(state: GraphState) -> Dict[str, List]:
    """
    Tool node

    Args:
        state (GraphState): Graph state

    Returns:
        Dict[str, List]: Updates to the `messages` attribute of graph state
    """
    result = []
    # Get the list of tool calls from messages
    tool_calls = state["messages"][-1].tool_calls
    # A tool_call looks as follows:
    # {
    #     "name": "get_information_for_question_answering",
    #     "args": {"user_query": "What are Atlas Triggers"},
    #     "id": "call_H5TttXb423JfoulF1qVfPN3m",
    #     "type": "tool_call",
    # }
    # Iterate through `tool_calls`
    for tool_call in tool_calls:
        # Run the `create_booking` tool only if the user approved it
        if tool_call["name"] == "create_booking" and not state.get("approved"):
            result.append(
                ToolMessage(
                    content=(
                        "The traveller did NOT approve this booking. Nothing was saved. "
                        "Ask them what they would like to change."
                    ),
                    tool_call_id=tool_call["id"],
                )
            )
            continue
        # Get the tool from `tools_map` defined in Step 4, using the `name` attribute of the `tool_call`
        tool = tools_map[tool_call["name"]]
        # Invoke the `tool` using the `args` attribute of the `tool_call`
        # HINT: See previous line to see how to extract attributes from `tool_call`
        observation = <CODE_BLOCK_11>
        # Append the result of executing the tool to the `result` list as a ToolMessage
        # The `content` of the message is `observation` i.e. result of the tool call
        # The `tool_call_id` can be obtained from the `tool_call`
        result.append(ToolMessage(content=str(observation), tool_call_id=tool_call["id"]))
    # Write `result` to the `messages` attribute of the graph state
    return {"messages": result}

In [ ]:
# Define the human-in-the-loop node
def confirm_node(state: GraphState) -> Dict:
    """
    Human-in-the-loop node. Pauses for human approval and records the decision to the graph state.

    Args:
        state (GraphState): Graph state

    Returns:
        Dict: Updates to the `approved` attribute of the graph state
    """
    # Get all the `create_booking` tool calls from the last message in the graph state
    bookings = [
        call
        for call in state["messages"][-1].tool_calls
        if call["name"] == "create_booking"
    ]
    # Suspend the graph and surface the booking(s) for review
    answer = interrupt(
        {
            "question": "Create this booking? Reply with y or n.",
            "bookings": [call["args"] for call in bookings],
        }
    )
    # Write the user's approval decision to the `approved` attribute of the graph state
    return {"approved": str(answer).strip().lower() == "y"}

# Step 9: Define conditional edges

In [ ]:
from langgraph.graph import END

In [ ]:
# Define conditional routing function
def route_tools(state: GraphState):
    """
    Route to the `confirm` node if the agent wants to create a booking, to the `tools` node otherwise, and to END if there are no tool calls.
    """
    # Get messages from graph state
    messages = state.get("messages", [])
    if len(messages) > 0:
        # Get the last AI message from messages
        ai_message = messages[-1]
    else:
        raise ValueError(f"No messages found in input state to tool_edge: {state}")
    # If there are no tool calls, route to END
    if not hasattr(ai_message, "tool_calls") or len(ai_message.tool_calls) == 0:
        return END
    # If the last AI message contains a `create_booking` tool call, route to the `confirm` node
    if any(call["name"] == "create_booking" for call in ai_message.tool_calls):
        return "confirm"
    # For any other tool calls, route to the `tools` node
    return "tools"

# Step 10: Build the graph

In [ ]:
from langgraph.graph import StateGraph, START

In [ ]:
# Instantiate the graph
graph = StateGraph(GraphState)

📚 https://docs.langchain.com/oss/python/langgraph/graph-api#nodes

In [ ]:
# Add nodes to the `graph` using the `add_node` function
# Add the `agent` node. The `agent` node should run the `agent_node` function
<CODE_BLOCK_12>
# Add the `tools` node. The `tools` node should run the `tool_node` function
<CODE_BLOCK_13>
# Add the `confirm` node. The `confirm` node should run the `confirm_node` function
<CODE_BLOCK_14>

📚 https://docs.langchain.com/oss/python/langgraph/graph-api#normal-edges

In [ ]:
# Add fixed edges to the `graph` using the `add_edge` method
# Add an edge from the START node to the `agent` node
<CODE_BLOCK_15>
# Add an edge from the `tools` node to the `agent` node
<CODE_BLOCK_16>
# Add an edge from the `confirm` node to the `tools` node
<CODE_BLOCK_17>

📚 https://docs.langchain.com/oss/python/langgraph/graph-api#conditional-edges

In [ ]:
# Use the `add_conditional_edges` method to add conditional edges from the `agent` node
# based on the output of the `route_tools` function
<CODE_BLOCK_18>

# Step 11: Add state persistence to the graph

In [ ]:
from langgraph.checkpoint.mongodb import MongoDBSaver

📚 https://www.mongodb.com/docs/atlas/ai-integrations/langgraph/#usage

In [ ]:
# Initialize a MongoDB checkpointer to persist graph state
# Use the mongodb_client` defined in Step 1
checkpointer = <CODE_BLOCK_19>

📚 https://www.mongodb.com/docs/atlas/ai-integrations/langgraph/#usage

In [ ]:
# Compile the `graph` from Step 10 with the `checkpointer`
app = <CODE_BLOCK_20>

In [ ]:
# Visualize the graph
app

# Step 12: Execute the graph

📚 https://docs.langchain.com/oss/python/langgraph/persistence#threads

In [ ]:
# Define a function to execute the graph and stream outputs from each step
def execute_graph(thread_id: str, user_input: str = None, approval: str = None) -> None:
    """
    Send a message to the agent, or answer a pending approval request

    Args:
        thread_id (str): Thread ID for the checkpointer
        user_input (str): User query string
        approval (str): "y" to approve a booking, "n" to decline
    """
    # Create a runtime config containing the thread ID
    config = <CODE_BLOCK_21>
    # The input to the graph is either a user query or an approval response
    # Answering an approval request sends a `Command` instead of a message
    input = (
        Command(resume=approval)
        if approval is not None
        else {"messages": [{"role": "user", "content": user_input}]}
    )
    for step in app.stream(input, config, stream_mode="values"):
        # When the graph pauses, the approval request shows up under `__interrupt__`
        if "__interrupt__" in step:
            request = step["__interrupt__"][0].value
            print("APPROVAL NEEDED:", request["question"])
            for booking in request["bookings"]:
                print("   ", ", ".join(f"{k}={v}" for k, v in booking.items()))
        else:
            step["messages"][-1].pretty_print()


In [ ]:
# Test the graph execution to view end-to-end flow
execute_graph(
    "1",
    "I want to go from NYC to Barcelona. What's the cheapest nonstop flight, and the cheapest place to stay for 2 people?",
)

In [ ]:
# Ask the agent to create a booking
# Notice the graph pauses for human approval
# Also notice that the agent remembers the context of the previous conversation and uses it to create the booking
execute_graph("1", "Book that flight and accommodation for 3 nights under the names Jane and John Doe.")

In [ ]:
# Approve the booking and watch the agent create the booking using the `create_booking` tool
execute_graph("1", approval="y")

In [ ]:
# Ask the same question on a different thread
# State is scoped to a `thread_id` and isn't persisted across threads.
execute_graph("2", "I want to go from NYC to Barcelona. What's the cheapest nonstop flight, and the cheapest place to stay for 2 people?")

# 🦹‍♀️ Step 13: Add long-term memory to the agent

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.mongodb import MongoDBStore, create_vector_index_config
from langchain_voyageai import VoyageAIEmbeddings
import uuid

📚 https://www.mongodb.com/docs/atlas/ai-integrations/langgraph/#usage-1

In [ ]:
# Initialize a MongoDB long-term memory store with Voyage AI embeddings to retrieve memories using vector search
mongodb_store = MongoDBStore(
    collection=memory_collection,
    index_config=create_vector_index_config(
        embed=VoyageAIEmbeddings(model="voyage-4"),
        dims=1024,
        fields=["content"]
    ),
)

In [ ]:
# Create a tool to save memories to the long-term memory store
@tool
def save_memory(memory: str, config: RunnableConfig) -> str:
    """
    Save important facts and preferences about the user for future conversations.

    Args:
    memory: The information to remember
    """
    user_id = config["configurable"]["user_id"]
    mongodb_store.put(
        # Namespace for the memory entry. You can also have sub-namespaces to store different types of memories, eg: (user_id, "preferences"), (user_id, "facts") etc.
        # Has to be a tuple, even if it contains empty values
        (user_id,),
        # Unique memory ID
        key=str(uuid.uuid4()),
        # Content of the memory- needs to be a dictionary
        value={"content": memory},
    )
    return f"Memory saved: {memory}"

In [ ]:
# Create a memory prompt providing instructions on what memories to extract and how to use the `save_memory` tool
MEMORY_PROMPT = """
Whenever the user states a preference, constraint, or fact about themselves, such as budget, party size, accommodation type etc, call `save_memory` to record it to long-term memory.
Save one fact/preference per memory entry. Do not save the same fact/preference multiple times.
Use past user preferences to personalize future conversations.
Past user memories:\n{memories}
"""

In [ ]:
# Update the tools list to include the `save_memory` tool
tools.append(save_memory)
tools_map = {t.name: t for t in tools}
# Bind the updated tool list to the `llm`
bind_tools = llm.bind_tools(tools)
# Update the prompt template to include the `MEMORY_PROMPT`
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT + MEMORY_PROMPT),
        MessagesPlaceholder(variable_name="messages")
    ]
)
# Pre-fill `top_k` and `tool_names` in the prompt template
prompt = prompt.partial(top_k=5, tool_names=", ".join([tool.name for tool in tools]))
# Chain the `prompt` with the tool-augmented LLM
llm_with_tools = prompt | bind_tools

In [ ]:
# Update the agent node to retrieve relevant long-term memories when generating a response
def agent_node(state: GraphState, config: RunnableConfig) -> Dict[str, List]:
    """
    Agent node

    Args:
        state (GraphState): Graph state
        config (RunnableConfig): Runtime config containing the `user_id` for memory retrieval

    Returns:
        Dict[str, List]: Updates to the `messages` attribute of graph state
    """
    # Get `messages` from the graph `state`
    messages = state["messages"]
    # Get the `user_id` from the runtime `config`
    user_id = config["configurable"]["user_id"]
    # Get the most recent user message
    # The last message in the state is a tool result when the agent loops back to this node after calling tools
    user_message = next(m.content for m in reversed(messages) if isinstance(m, HumanMessage))
    # Search for long-term memories that are relevant to the user's message
    memories = mongodb_store.search((user_id,), query=user_message, limit=10)
    # Format retrieved memories into a string
    memories = "\n".join(f"- {m.value['content']}" for m in memories) or "No memories yet."
    # Invoke the tool-augmented LLM with the `memories` and `messages` (includes chat history) to generate a response
    result = llm_with_tools.invoke(
        {
            "memories": memories,
            "messages": messages,
        }
    )
    # Write the `result` to the `messages` attribute of the graph state
    return {"messages": [result]}

In [ ]:
# Update the tool node to pass the runtime config to tools so they can access the `user_id`
def tool_node(state: GraphState, config: RunnableConfig) -> Dict[str, List]:
    """
    Tool node

    Args:
        state (GraphState): Graph state
        config (RunnableConfig): Runtime config
    Returns:
        Dict[str, List]: Updates to the `messages` attribute of graph state
    """
    result = []
    # Get the list of tool calls from messages
    tool_calls = state["messages"][-1].tool_calls
    # Iterate through `tool_calls`
    for tool_call in tool_calls:
        # Run the `create_booking` tool only if the user approved it
        if tool_call["name"] == "create_booking" and not state.get("approved"):
            result.append(
                ToolMessage(
                    content=(
                        "The traveller did NOT approve this booking. Nothing was saved. "
                        "Ask them what they would like to change."
                    ),
                    tool_call_id=tool_call["id"],
                )
            )
            continue
        # Get the tool from `tools_map` defined in Step 4, using the `name` attribute of the `tool_call`
        tool = tools_map[tool_call["name"]]
        # Invoke the `tool` using the `args` attribute of the `tool_call` and the runtime `config`
        observation = tool.invoke(tool_call["args"], config)
        # Append the result of executing the tool to the `result` list as a ToolMessage
        # The `content` of the message is `observation` i.e. result of the tool call
        # The `tool_call_id` can be obtained from the `tool_call`
        result.append(ToolMessage(content=str(observation), tool_call_id=tool_call["id"]))
    # Write `result` to the `messages` attribute of the graph state
    return {"messages": result}

In [ ]:
# Rebuild the agent graph with the updated `agent_node` and `tool_node`
graph = StateGraph(GraphState)
# Add the `agent`, `tools` and `confirm` nodes as before
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)
graph.add_node("confirm", confirm_node)

# Add the fixed edges as before
graph.add_edge(START, "agent")
graph.add_edge("tools", "agent")
graph.add_edge("confirm", "tools")

# Add the conditional edges from the `agent` node as before
graph.add_conditional_edges(
    "agent",
    route_tools,
    {"tools": "tools", "confirm": "confirm", END: END},
)


In [ ]:
# Compile the graph with the MongoDB checkpointer for state persistence and short-term memory, and the MongoDB memory store for long-term memory
app = graph.compile(checkpointer=checkpointer, store=mongodb_store)

In [ ]:
# Update the execution function to take user ID as input to organize state and long-term memory by user
def execute_graph(
    thread_id: str, user_id: str, user_input: str = None, approval: str = None
) -> None:
    """
    Send a message to the agent, or answer a pending approval request

    Args:
        thread_id (str): Thread ID for the checkpointer
        user_id (str): User ID to organize state and long-term memory
        user_input (str): User query string
        approval (str): "y" to approve a booking, "n" to decline
    """
    # Create a runtime config containing the thread ID and user ID
    # Scope the `thread_id` by user as well to prevent collisions across users
    config = {"configurable": {"thread_id": f"{user_id}-{thread_id}", "user_id": user_id}}
    # The input to the graph is either a user query or an approval response
    # Answering an approval request sends a `Command` instead of a message
    input = (
        Command(resume=approval)
        if approval is not None
        else {"messages": [{"role": "user", "content": user_input}]}
    )
    for step in app.stream(input, config, stream_mode="values"):
        # When the graph pauses, the approval request shows up under `__interrupt__`
        if "__interrupt__" in step:
            request = step["__interrupt__"][0].value
            print("APPROVAL NEEDED:", request["question"])
            for booking in request["bookings"]:
                print("   ", ", ".join(f"{k}={v}" for k, v in booking.items()))
        else:
            step["messages"][-1].pretty_print()


In [ ]:
# Test creating memories in thread `1` for user `Jane`
# Notice the `save_memory` tool being invoked to save the user's preferences
execute_graph(
    "1",
    "Jane",
    "Remember that I only want entire homes and never spend more than $100 a night on stays."
)

In [ ]:
# Test cross-session long-term memory persistence by starting a new thread `2` for user `Jane`
# Notice that the agent is able to recall the user's preferences from the thread `1`
execute_graph(
    "2",
    "Jane",
    "I want to go to Barcelona. Can you find me places to stay?"
)

In [ ]:
# Test that memories aren't leaking across users-- Jane's preferences should not be used for user `John`
# Notice that the agent does NOT use Jane's preferences to answer John's question
execute_graph(
    "1",
    "John",
    "I want to go to Barcelona. Can you find me places to stay?"
)